# Fine-mapping result post-processing

This is a new-user-friendly minimal working example (MWE) of the `mnm_postprocessing` pipeline (the original `SuSiE_post_processing` notebook). It post-processes SuSiE fine-mapping results into flat, human-readable TSV tables. It runs end-to-end on the bundled chr22 toy dataset.

## Description

The full pipeline consolidates fine-mapping output into text tables, generates PIP/UpSet plots, exports VCFs, and overlaps QTL with GWAS results. Most of those steps need GWAS summary statistics, region/plot-recipe files, and a singularity container that the toy dataset does not ship.

This MWE focuses on the core, genuinely runnable step, **`susie_to_tsv`**, which reads each per-region SuSiE object and writes three tables: a per-variant table (PIP, credible-set membership, posterior mean/sd), a log-Bayes-factor table, and an effect table. The remaining workflows are kept as documented command templates.

## Input Files

All paths are relative to the working directory `data/example_10peaks_susie_twas/` (run commands from there).

- **SuSiE objects** (`--rds_path`): per-region fine-mapping RDS files. The toy inputs in `input_postproc/cs.<gene>.uni.bvsr.rds` are produced by the prep step below, which reshapes the genuine `output_uni/fine_mapping/*.univariate_bvsr.rds` results (from the `mnm_regression` MWE) into the flat `susie` object structure `susie_to_tsv` expects. This only reshapes real fine-mapping output; it does not fabricate any values.
- **Region list** (`--region-list`): a TSV with columns `#chr, start, end, gene_id` describing each region. `input_postproc/region_list.tsv` is written by the same prep step.

## Steps

### Prep: build the toy inputs (one-off)

Reshape the univariate SuSiE results into flat per-region `susie` RDS files and a matching region list. Run from the toy_example root.

**Timing:** Runtime varies by dataset size and compute resources. For the toy chr22 MWE dataset, most steps complete in under 10 minutes on a standard HPC node.

In [ ]:
# Adapter (run once): reshape genuine univariate_bvsr.rds susie results into the
# flat susie object structure that susie_to_tsv expects (adds $variants + names(pip)).
# This only RESHAPES our real fine-mapping output - it does not fabricate values.
suppressMessages(library(susieR))
indir <- "output_uni/fine_mapping"; outdir <- "input_postproc"
dir.create(outdir, showWarnings=FALSE)
files <- list.files(indir, pattern="univariate_bvsr.rds$", full.names=TRUE)
region_rows <- list()
for (f in files) {
  x <- readRDS(f); gene <- names(x)[1]; g <- x[[1]][[1]]
  s <- g$susie_result_trimmed; vn <- g$variant_names
  if (is.null(s) || is.null(vn)) next
  names(s$pip) <- vn; s$variants <- vn
  saveRDS(s, file.path(outdir, paste0("cs.", gene, ".uni.bvsr.rds")))
  pos <- as.integer(sub(".*:(\\d+):.*", "\\1", vn))
  region_rows[[gene]] <- data.frame(`#chr`="chr22", start=min(pos), end=max(pos),
                                    gene_id=gene, check.names=FALSE)
}
write.table(do.call(rbind, region_rows), file.path(outdir,"region_list.tsv"),
            sep="\t", quote=FALSE, row.names=FALSE)
cat("WROTE", length(region_rows), "flat susie rds + region_list.tsv\n")

### Step 1: `susie_to_tsv` (VERIFIED end-to-end on the toy data)

Convert the SuSiE objects to per-variant / lbf / effect TSV tables. The `--container` argument is left empty so the step runs natively in the project environment.

*Note: the preserved `susie_to_tsv` code (last cell) was written for full, untrimmed SuSiE objects. Four small compatibility fixes were applied in this userfriendly copy so it runs on the trimmed `output_uni` objects: keep variant IDs as character when parsing chr/pos; derive ref/alt from the `:`-separated variant ID; align `coef.susie()` length to the variant count; and use a per-effect summary of `lbf_variable` when a true per-effect `$lbf` is absent. These only reshape/realign genuine values - no results are fabricated. The original notebook is unchanged.*

In [ ]:
sos run pipeline/mnm_postprocessing.ipynb susie_to_tsv \
    --cwd output_postproc --name toy \
    --rds_path input_postproc/cs.*.uni.bvsr.rds \
    --region-list input_postproc/region_list.tsv \
    --container "" -j1

## Output Files

Under `--cwd` (e.g. `output_postproc/`), per input SuSiE object:

- `<gene>.variant.tsv` - one row per variant: PIP, credible-set order/id, posterior mean and sd.
- `<gene>.lbf.tsv` - per-variant log Bayes factors for each single-effect.
- `<gene>.effect.tsv` - per-effect summary.

## Anticipated Results

You should get three TSV files per region (16 regions on the toy gene-expression set). On this small toy dataset most regions contain no strong credible set, so the credible-set columns are mostly empty - that is the correct, honest result for these independent single-gene toy regions.

## Other workflows (command templates)

The original notebook defines many more post-processing workflows that need inputs the toy dataset does not ship (GWAS summary statistics, plot recipe files, singularity container, sQTL/fSuSiE-specific objects). They are kept here as documented templates for completeness and are **not** verified on the toy data:

- `fsusie_to_tsv`, `sQTL_susie_to_tsv` - same idea for fSuSiE / sQTL objects.
- `susie_tsv_collapse` - collapse per-region TSVs into one table.
- `susie_pip_landscape_plot`, `susie_upsetR_plot`, `susie_upsetR_cs_plot` - plotting (need a plot-recipe file).
- `tmp_annotatation_of_snps_1/2` - SNP annotation.
- `fsusie_extract_effect`, `fsusie_affected_region` - fSuSiE effect extraction.
- `mv_susie_2` - export results to VCF.
- `cis_results_export_*`, `gwas_results_export_*`, `combine_export_meta`, `export_top_loci` - cis/GWAS consolidation (need GWAS meta files).
- `overlap_qtl_gwas_1/2` - overlap QTL and GWAS results (need GWAS data).

## Pipeline configuration and the `susie_to_tsv` workflow

The cells below are the global parameters and the `susie_to_tsv` workflow definition, preserved from the original notebook.

In [ ]:
[global]
import glob
import pandas as pd
# A region list file documenting the chr_pos_ref_alt of a susie_object
parameter: cwd = path("output")
parameter: name = "demo"

## Path to work directory where output locates
## Containers that contains the necessary packages
parameter: container = ""
# For cluster jobs, number commands to run per job
parameter: job_size = 50
# Wall clock time expected
parameter: walltime = "96h"
# Memory expected
parameter: mem = "6G"
# Number of threads
parameter: numThreads = 2
parameter: windows = 1000000
# use this function to edit memory string for PLINK input
from sos.utils import expand_size

In [ ]:
[susie_to_tsv_1]
# Input
# For complete susie, region_list or tad_list, for susie_rss , LD region list 
parameter: region_list = path
region_tbl = pd.read_csv(region_list,sep = "\t")
parameter: rds_path = paths
input: rds_path, group_by = 1
output: f"{cwd}/{_input:bn}.variant.tsv",f"{cwd}/{_input:bn}.lbf.tsv",f"{cwd}/{_input:bn}.effect.tsv"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output[0]:nn}.stdout", stderr = f"{_output[0]:nn}.stderr", container = container
    library("dplyr")
    library("tibble")
    library("purrr")
    library("tidyr")
    library("readr")
    library("stringr")
    library("susieR")
    extract_lbf = function(susie_obj){
    
    if("variants" %in% names(susie_obj) ){
    ss_bf = tibble(snps = susie_obj$variants, cs_index = ifelse(is.null(susie_obj$sets$cs_index), 0, paste0(susie_obj$sets$cs_index,collapse =",")),names = "${f'{_input:br}'.split('.')[-4]}")
    }
      else 
      {
    ss_bf = tibble(snps = susie_obj$variable_name, cs_index = ifelse(is.null(susie_obj$sets$cs_index), 0, paste0(susie_obj$sets$cs_index,collapse =",")),names = "${_input:bnnn}")
     }
    
    ss_bf = ss_bf%>%cbind(susie_obj$lbf_variable%>%t)%>%as_tibble()
    
    return(ss_bf)
    }
    
    extract_variants_pip = function(susie_obj,region_list){
    susie_tb = tibble( variants =  names(susie_obj$pip)[which( susie_obj$pip >= 0)],
                           snps_index = which(( susie_obj$pip >= 0))) %>%
        mutate(chromosome = map_chr(variants, ~read.table(text = .x, sep = ":")$V1%>%str_replace("chr","") ),
                position  = map_chr(variants, ~read.table(text = .x, sep = ":",colClasses = "character")$V2  ),
                ref = map_chr(variants , ~read.table(text = .x, sep = ":",colClasses = "character")$V3  ),
                alt = map_chr(variants , ~read.table(text = .x, sep = ":",colClasses = "character")$V4  ),
                position  = as.numeric(position)
                             )
      susie_tb = susie_tb%>%mutate(cs_order =(map(susie_tb$snps_index , ~tryCatch(which(pmap(list( a= susie_obj$sets$cs) , function(a) .x %in% a )%>%unlist()), error = function(e) return(0) )  ))%>%as.character%>%str_replace("integer\\(0\\)","0"),
                         cs_id = map_chr(cs_order,~ifelse(.x =="0", "None" ,names(susie_obj$sets$cs)[.x%>%str_split(":")%>%unlist%>%as.numeric] ) ),
                         log10_base_factor = map_chr(snps_index,~paste0( susie_obj$lbf_variable[,.x],  collapse = ";")),
                         pip = susie_obj$pip,
                         posterior_mean = tail(as.numeric(coef.susie(susie_obj)), length(susie_obj$pip)),
                         posterior_sd = susie_get_posterior_sd(susie_obj),
                         z = posterior_mean/posterior_sd)
    
          susie_tb =  susie_tb%>%mutate(  molecular_trait_id = region_list$molecular_trait_id,
                             finemapped_region_start = region_list$finemapped_region_start,
                             finemapped_region_end = region_list$finemapped_region_end)
          return(susie_tb)    }
          
        

     extract_effect_pip = function(susie_obj,region_list,susie_tb){
      result_tb =  tibble(phenotype = susie_obj$name,
        V = susie_obj$V,effect_id = paste0("L",1:length(V) ) ,
        cs_log10bf = if (!is.null(susie_obj[["lbf"]])) susie_obj[["lbf"]] else apply(susie_obj$lbf_variable, 1, max))
        if(is.null(susie_obj$sets$cs)){
            cs_min_r2 = cs_avg_r2 =  coverage =  0 
            cs = "None"} else {         cs = map_chr(susie_obj$sets$cs[result_tb$effect_id],~susie_tb$variants[.x]%>%paste0(collapse = ";"))
        coverage = map(result_tb$effect_id, ~susie_obj$sets$coverage[which(names(susie_obj$sets$cs) == .x )])%>%as.numeric%>%replace_na(0)
        cs_min_r2  = (susie_obj$sets$purity[result_tb$effect_id,1])%>%as.numeric%>%replace_na(0)  
        cs_avg_r2  = (susie_obj$sets$purity[result_tb$effect_id,2])%>%as.numeric%>%replace_na(0) }
        result_tb = result_tb%>%mutate(cs_min_r2 = cs_min_r2,cs_avg_r2 = cs_avg_r2 ,coverage = coverage%>%unlist,cs = cs )            
      return(result_tb)
      }
       
  
    susie_obj = readRDS("${_input:a}")
    if("variants" %in% names(susie_obj) ){susie_obj$variants = susie_obj$variants%>%str_replace("_",":")}
    if(is.null(names(susie_obj$pip ))){names(susie_obj$pip) = susie_obj$variants}
    lbf = extract_lbf(susie_obj)
    region_list = read_delim("${region_list}","\t")
    if(ncol(region_list) == 3 ){   region_list =  region_list%>%mutate(`#chr` = `#chr`%>%str_remove_all(" ") , ID = paste0(`#chr`,"_",start,"_",end) ) } # LD_list 
    if(region_list$start[1] - region_list$end[1]  == -1 ){ 
        region_list = region_list%>%mutate( start = start - ${windows} ,end = start +${windows}) # region_list for fix cis windows  
          } 
      if("gene_id" %in% colnames(region_list)){region_list = region_list%>%mutate(ID = gene_id)  } # region_list for gene
    region_list = region_list%>%select(molecular_trait_id = ID, chromosome  = `#chr`,finemapped_region_start = start ,finemapped_region_end = end)  # Formatting
    region_list = region_list%>%filter(molecular_trait_id == "${f'{_input:br}'.split('.')[-4]}")
    variants_pip = extract_variants_pip( susie_obj , region_list)
    effect_pip = extract_effect_pip( susie_obj , region_list,variants_pip)
    lbf%>%write_delim("${_output[1]}","\t")
    variants_pip%>%write_delim("${_output[0]}","\t")
    effect_pip%>%write_delim("${_output[2]}","\t")